In [61]:
import pandas as pd 
import re
import os
import json
pathreal = '/home/sonia/be_great/data/adult/latest'
path = '/mnt/data/sonia/ckpts/adult/dgpt2/plain-aug16/'

raws = []
with open(os.path.join(path, 'samples.txt'), 'r') as f:
    for raw in f.readlines():
        raws.append(re.sub('is\?', 'is ?', raw))

real = pd.read_csv(os.path.join(pathreal, 'all.csv'))
cols  = set(real.columns)
 
def parse_line(l):
    entries = l[:-1].split('.<EOS>') # remove newline at end
    # print(entries)
    words = [c.split(' ') for c in entries] #'name', 'is', 'value'
    # print(words)
    d = {c[0]:c[2] for c in words if len(c)==3 and c[0] in cols}
    if set(d.keys()) == cols:
        return d 
    else:
        return None

line_dicts = [parse_line(l) for l in raws]
line_dicts = [l for l in line_dicts if l is not None]
print(len(raws)-len(line_dicts), 'problem lines')
df = pd.DataFrame.from_records(line_dicts)

with open(os.path.join(pathreal, 'config.json'), 'r') as f:
    dataconfig = json.load(f)
ords = dataconfig['ords']

ordvals = {col:set(real[col].unique()) for col in ords}
for col in ordvals:
    ordvals[col] = [str(val).strip() for val in ordvals[col]]

for col in ordvals:
    df = df[df[col].isin(ordvals[col])]
    print(col, len(df))
    
df.to_csv(os.path.join(path, 'samplesclean.csv'), index=False)

4143 problem lines
class 5527
education 5245
marital-status 4754
occupation 4680
relationship 4573
race 4241
sex 4084
native-country 3986
